# Comparing MLP and Convolutional Models

In this week's homework, we'll use PyTorch to compare the performance of multi-layer perceptrons (MLP's; the kind of model we've looked at so far) against _convolutional_ neural networks (CNN's). We'll talk about convolutions in class on Monday, but you can get started on the MLP part of the homework as soon as you're ready. For our comparison we'll go back to the CIFAR-10 dataset, since it's a bit less chaotic than CIFAR-100.

As with the last homework, I have some guidelines about what parts of the homework are necessary for different grades:

- The basic version, for a C, is to define and train an MLP and a CNN.
- On top of that, the B level work requires you to analyze your results a bit. I'll describe this in more detail later in the notebook after the code that sets up and trains the networks.
- For an A, you'll need to finish the activation map visualization in the last section of this notebook. More details on that later on.

The rest of the document is organized into sections which are labeled with the grade they correspond to.

## Data Setup

The first section of the notebook gets the dataset and sets up the transforms we need. The code in this section is complete, although you may need to change the dataset path or change the `transform` definition to match your version of torchvision.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2 as transforms

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineRenderer.figure_format = 'retina'

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

mps


In [2]:
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ConvertImageDtype(),
    # Depending on your torchvision version you may need to change these:
    # - If you don't have torchvision.transforms.v2, then import torchvision.transforms
    #   instead and use ToTensor() to replace _both_ of the transforms above.
    # - If you have v2 but it says ToImage() is undefined, then use ToImageTensor() instead.
])

# If you already have the CIFAR10 data downloaded from the in-class notebook, you can change the path here
# to point to it so you avoid downloading a second copy.
cifar = torchvision.datasets.CIFAR10("../data/torch/cifar", download=True, transform=transform)
train_size = int(0.8 * len(cifar))
train_data, valid_data = torch.utils.data.random_split(cifar, [train_size, len(cifar) - train_size])

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(len(cifar))

Files already downloaded and verified
50000


In [3]:
mean = []
for x, _ in cifar:
    mean.append(torch.mean(x, dim=(1, 2)))
mean = torch.stack(mean, dim=0).mean(dim=0)
std = []
for x, _ in cifar:
    std.append(((x - mean[:,np.newaxis,np.newaxis]) ** 2).mean(dim=(1, 2)))
std = torch.stack(std, dim=0).mean(dim=0).sqrt()
print(mean, std)

tensor([0.4914, 0.4822, 0.4465]) tensor([0.2470, 0.2435, 0.2616])


In [4]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

normalize = transforms.Normalize(cifar_mean, cifar_std)

## Models and Training (C)

First, define an MLP model for the CIFAR dataset. An MLP, also called a fully-connected network, consists of linear computations alternated with nonlinear activation functions, just like every network we've looked at in this class so far. This is very similar to what we did in clas on Wednesday and lab on Friday.

In [5]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(3*32*32, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x.reshape(-1, 3*32*32)).squeeze(dim=0)

Now let's define a training function for our MLP. As usual, you may want to add more arguments to the training function. For the latter parts of the notebook, it will be helpful if your training function returns both the model and a list of the training and validation accuracies for each epoch.

In [6]:
def train(model_class=MLP, lr=1e-3, epochs=10, batch_size=64, reg=0):
 
    data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size, shuffle=False)

    train_accs = []
    valid_accs = []
    
    network=model_class().to(device)

    loss = nn.CrossEntropyLoss()

    opt = optim.SGD(network.parameters(), momentum=0.9, lr=lr, weight_decay=reg)

    for i in range(epochs):
        
        accs = []
        for batch_xs, batch_ys in data_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device) 
            batch_xs = normalize(batch_xs)
            preds = network(batch_xs)
            loss_val = loss(preds, batch_ys)
            accs.append((preds.argmax(dim=1) == batch_ys).float().mean())

            opt.zero_grad()
            loss_val.backward()
            opt.step()

        train_accs.append(torch.tensor(accs).mean().item()) 
        accs = []
        for batch_xs, batch_ys in valid_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device)
            preds = network(normalize(batch_xs))
            accs.append((preds.argmax(dim=1) == batch_ys).float().mean())
        valid_accs.append(torch.tensor(accs).mean().item())
        print("Epoch:", i, "Valid accuracy:", valid_accs[-1])

    return network, train_accs, valid_accs

In [7]:
%%time
mlp_model, mlp_train_accs, mlp_valid_accs = train(model_class=MLP, lr=5e-3, epochs=15, batch_size=64)
print (mlp_train_accs)

Epoch: 0 Valid accuracy: 0.44098326563835144
Epoch: 1 Valid accuracy: 0.4633758068084717
Epoch: 2 Valid accuracy: 0.5034832954406738
Epoch: 3 Valid accuracy: 0.5050756335258484
Epoch: 4 Valid accuracy: 0.5147293210029602
Epoch: 5 Valid accuracy: 0.5230891704559326
Epoch: 6 Valid accuracy: 0.5363256335258484
Epoch: 7 Valid accuracy: 0.5282643437385559
Epoch: 8 Valid accuracy: 0.5323447585105896
Epoch: 9 Valid accuracy: 0.5196058750152588
Epoch: 10 Valid accuracy: 0.5221934914588928
Epoch: 11 Valid accuracy: 0.5316480994224548
Epoch: 12 Valid accuracy: 0.5293591022491455
Epoch: 13 Valid accuracy: 0.518909215927124
Epoch: 14 Valid accuracy: 0.5228901505470276
[0.38874998688697815, 0.4761750102043152, 0.5187000036239624, 0.5440750122070312, 0.5724499821662903, 0.5903000235557556, 0.6141999959945679, 0.6345750093460083, 0.6542500257492065, 0.6701750159263611, 0.6861749887466431, 0.7013499736785889, 0.7199000120162964, 0.7349249720573425, 0.7525249719619751]
CPU times: user 1min 40s, sys: 13

Now you can define a CNN for the same task.

In [11]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 8, (7, 7), padding=3),
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.AvgPool2d(kernel_size=(4, 4)),
            nn.Flatten(),
            nn.Linear(32*4*4, 10)
        )

    def forward(self, x):
        return self.model(x)

Now you can train your CNN. You should be able use the same `train` function but pass it `model_class=CNN` (you may need to change some other hyperparameters to get good results).

In [12]:
%%time
cnn_model, cnn_train_accs, cnn_valid_accs = train(model_class=CNN, lr=5e-3, epochs=15, batch_size=64)

Epoch: 0 Valid accuracy: 0.4530254900455475
Epoch: 1 Valid accuracy: 0.49442675709724426
Epoch: 2 Valid accuracy: 0.5195063948631287
Epoch: 3 Valid accuracy: 0.5586186051368713
Epoch: 4 Valid accuracy: 0.5791202187538147
Epoch: 5 Valid accuracy: 0.5918591022491455
Epoch: 6 Valid accuracy: 0.6128582954406738
Epoch: 7 Valid accuracy: 0.6206210255622864
Epoch: 8 Valid accuracy: 0.6056926846504211
Epoch: 9 Valid accuracy: 0.6049960255622864
Epoch: 10 Valid accuracy: 0.6324641704559326
Epoch: 11 Valid accuracy: 0.6398288011550903
Epoch: 12 Valid accuracy: 0.6433120965957642
Epoch: 13 Valid accuracy: 0.6336584687232971
Epoch: 14 Valid accuracy: 0.640824019908905
CPU times: user 1min 48s, sys: 14 s, total: 2min 2s
Wall time: 1min 31s


Given how long it takes to train the CNN model, now might be a good time to talk about saving and loading models. The cell below will save your CNN model to the file `cnn_model.pt`.

In [13]:
torch.save(cnn_model.state_dict(), "cnn_model.pt")

If you have the file `cnn_model.pt` on your system, then you can load it with this code:

In [14]:
loaded_model = CNN()
loaded_model.load_state_dict(torch.load("cnn_model.pt"))

/var/folders/5x/3tpx9mmn5lvfc0rdtwv4sxnw0000gn/T/ipykernel_9577/1420923919.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load("cnn_m

<All keys matched successfully>

## Analysis (B)

In this section, we'll do some minor analysis of the results of our experiment above. Let's compare

- the time taken to train each model,
- the validation accuracy over time,
- the total number of parameters in each model.

That last measurement, the total number of parameters, is something you'll need to compute based on your network architecture. When I say number of parameters here, I'm referring to the total number of dimensions in the parameter space. That is, the total number of individual real numbers in our parameters. For example, a matrix of size 100x10 has 1,000 parameters.

I'm not asking for anything specific here, I'm just looking for you to think about the models we're using and their relative merits. If you think of something else that might be useful to compare the two networks, mention that as well.

### My Analysis:

Below I wrote a quick for loop to compute the average validation accuracy of the MLP and the CNN models. Over 5 trials, the average validation accuracy for the MLP was approximately 0.5179, whereas the CNN was over 10% more accurate with a 0.6206 average. Each of these were trained with the same number of epochs (10) and the same batch size, though with learning rates that I deemed more optimal for each one (it gave minimal oscillation towards the end). The MLP had a learning rate of 1e-3, which is smaller than the learning rate of the CNN, 5e-3. When I tested 5e-3 on the MLP, the loss oscillated significantly. 

It is also relevant to note that the validation accuracy starts nearly 10% higher in the first epoch of the CNN than in the MLP. So, though they each become a similar amount of more accurate during training (12-15% more accurate), the CNN then ends with a higher final accuracy. I personally find this impressive given the total number of parameters in each model: the MLP having 820,480 paramaters compared to the CNN's 7,256. The architecture is much more simple and seems better designed for image classification to me in the way it uses the kernels to synthesize the data in each n x n square of pixels rather than needing the complexity of a fully connected network. 

In general, the CNN took about 122 seconds to train on my device, whereas the MLP took 113 seconds. This difference is trivial to me when one takes into consideration the noticeably higher validation accuracy given by the CNN. Especially considering each of my models are not necessarily that optimized for this task--there could be schedulers for the learning rate or different kernel sizes for the CNN that could be better, yet already with these quickly found values it is obviously the better choice. 

Given all of that information, the CNN seems to be the model that makes the most sense for instances of image classification such as this one. 

In [16]:
avg_mlp_acc = 0
avg_cnn_acc = 0

for i in range (5):
    mlp_model, mlp_train_accs, mlp_valid_accs = train(model_class=MLP, lr=1e-3, epochs=10, batch_size=64)
    avg_mlp_acc += mlp_valid_accs[-1]
    print("current mlp total: ", avg_mlp_acc)
    cnn_model, cnn_train_accs, cnn_valid_accs = train(model_class=CNN, lr=5e-3, epochs=10, batch_size=64)
    avg_cnn_acc += cnn_valid_accs[-1]
    print("current cnn total: ", avg_cnn_acc)
    

avg_mlp_acc /= 5
avg_cnn_acc /= 5
print(avg_mlp_acc, avg_cnn_acc)
# 0.5178941011428833 0.6206409335136414

Epoch: 0 Valid accuracy: 0.38276273012161255
Epoch: 1 Valid accuracy: 0.43192675709724426
Epoch: 2 Valid accuracy: 0.4583001732826233
Epoch: 3 Valid accuracy: 0.46924760937690735
Epoch: 4 Valid accuracy: 0.48298168182373047
Epoch: 5 Valid accuracy: 0.4958200752735138
Epoch: 6 Valid accuracy: 0.5054737329483032
Epoch: 7 Valid accuracy: 0.5042794346809387
Epoch: 8 Valid accuracy: 0.5169187784194946
Epoch: 9 Valid accuracy: 0.5242834687232971
current mlp total:  0.5242834687232971
Epoch: 0 Valid accuracy: 0.4706409275531769
Epoch: 1 Valid accuracy: 0.5104498267173767
Epoch: 2 Valid accuracy: 0.5568272471427917
Epoch: 3 Valid accuracy: 0.5638933181762695
Epoch: 4 Valid accuracy: 0.5909633636474609
Epoch: 5 Valid accuracy: 0.6005175113677979
Epoch: 6 Valid accuracy: 0.612957775592804
Epoch: 7 Valid accuracy: 0.6267914175987244
Epoch: 8 Valid accuracy: 0.624800980091095
Epoch: 9 Valid accuracy: 0.6363455653190613
current cnn total:  0.6363455653190613
Epoch: 0 Valid accuracy: 0.3826632201671

## Interpretation (A)

In this section, we'll explore our CNN and try to understand how it learns to recognize different objects. There are several approaches to this problem. For today we'll look at a simple one that only works for the first convolutional layer. If we plot the weights of the layer as image data, we can visualize the kinds of patterns the convolution is scanning for. In order to do this, you'll need to normalize each kernel to the range [0, 1], then transpose the axis to format the weights as image data, then use the `imshow` function from `matplotlib.pyplot`. The weights of a convolutional layer `conv` are stored as a tensor of shape `Cin, Cout, H, W` in `conv.weight`.

If you have a torch tensor `x` and you want to display it with `imshow` you'll need to convert it to numpy by calling `x.detach().numpy()`.

Once you've got the plots displaying, take a minute to think about convolutions and try to tell me what the plots mean in terms of how the network recognizes images. (I'm being deliberately a bit vague about this.)

In [ ]:
# If your first convolutional layer has fewer than 16 output channels then you'll need to
# change the number of plots here.
fig, axs = plt.subplots(4, 4, figsize=(5, 5))
for i in range(4):
    for j in range(4):
        c = 4 * i + j
        YOUR_CODE_HERE
        axs[i,j].imshow(YOUR_CODE_HERE)
plt.show()